In [9]:
import math

# CHECK / IMPORT MATPLOTLIB
try:
    import matplotlib.pyplot as plt
    print('Matplotlib imported successfully.')
except ImportError:
    print('Error! Matplotlib is not installed.')
    print('Install using: pip install matplotlib')
    exit()

# CHECK / IMPORT PANDAS
try:
    import pandas as pd
    print('Pandas imported successfully.')
except ImportError:
    print('Error! Pandas is not installed.')
    print('Install using: pip install pandas')
    exit()

Matplotlib imported successfully.
Pandas imported successfully.


## Objective 1: File Handling and Leveling Type Detection


In [10]:
class FileHandler:
    '''
    Manage CSV file input, validate the file format, and identify the leveling type.

    The class accepts only CSV files and checks whether the uploaded data follows
    either the 3-wire leveling column format or the differential leveling column
    format. It returns both the loaded dataframe and the detected leveling type.
    '''

    def __init__(self):
        '''
        Initialize file requirements, expected column names, and storage attributes.
        '''
        self.extension = '.csv'
        self.three_wire_columns = [
            'BS_point',
            'BS_upper',
            'BS_middle',
            'BS_lower',
            'FS_point',
            'FS_upper',
            'FS_middle',
            'FS_lower'
        ]
        self.differential_columns = [
            'BS_point',
            'BS_Reading',
            'BS_Dist',
            'FS_point',
            'FS_Readings',
            'FS_Dist'
        ]
        self.filename = ''
        self.leveling_data = None
        self.leveling_type = ''

    def read_file(self):
        '''
        Ask the user for a CSV file, read the file, and validate its structure.

        The method keeps asking for a filename until a valid and recognized CSV
        file is provided. It handles missing files, empty files, and invalid CSV
        formatting before returning the dataframe and leveling type.
        '''

        while True:
            self.filename = input('Enter name of csv file to open: ')

            # Automatically add .csv
            if not self.filename.endswith(self.extension):
                self.filename += self.extension

            try:
                self.leveling_data = pd.read_csv(self.filename)

            except FileNotFoundError:
                print('Error! File not found.')
                continue

            except pd.errors.EmptyDataError:
                print('Error! CSV file is empty.')
                continue

            except pd.errors.ParserError:
                print('Error! Invalid CSV formatting.')
                continue

            # Detect leveling type
            self.detect_leveling_type()

            # If valid format
            if self.leveling_type != '':
                break

        return self.leveling_data, self.leveling_type

    def detect_leveling_type(self):
        '''
        Determine whether the loaded CSV is 3-wire or differential leveling data.

        The method checks the dataframe column names against the required column
        lists. If the columns match one of the accepted formats, the leveling
        type is stored in self.leveling_type. Otherwise, an error message is
        displayed and the file is treated as invalid.
        '''

        columns = self.leveling_data.columns.tolist()

        # 3-WIRE
        if all(col in columns for col in self.three_wire_columns):
            self.leveling_type = '3-Wire'

            print(f'\nOpening {self.filename}')
            print('Detected: 3-Wire Leveling Data')

        # DIFFERENTIAL
        elif all(col in columns for col in self.differential_columns):
            self.leveling_type = 'Differential'

            print(f'\nOpening {self.filename}')
            print('Detected: Differential Leveling Data')

        # INVALID
        else:
            self.leveling_type = ''

            print('Error! CSV format is not recognized.')
            print('Check column names and file structure.')

In [11]:
# Create FileHandler object to manage CSV input and detection
handler = FileHandler()

# Read CSV file and get dataset + leveling type
data, level_type = handler.read_file()

Enter name of csv file to open:  leveling3wire_data



Opening leveling3wire_data.csv
Detected: 3-Wire Leveling Data


In [12]:
data.head()

,BS_point,BS_upper,BS_middle,BS_lower,FS_point,FS_upper,FS_middle,FS_lower
0,BM1,0.89,0.70,0.51,TP 1,2.18,1.99,1.80
1,TP 1,1.66,1.53,1.40,TP 2,1.06,0.93,0.80
2,TP 2,1.65,1.57,1.50,TP 3,1.05,1.02,0.94
3,TP 3,1.74,1.66,1.58,TP 4,1.10,1.02,0.94
4,TP 4,1.91,1.82,1.73,TP 5,0.93,0.84,0.74


## Objectives 2-4: Elevation, Misclosure, and Adjustment Computations


In [13]:
class LevelLoop:
    '''
    Perform leveling loop computations for elevation, misclosure, and adjustment.

    The class works with either 3-wire or differential leveling data. It computes
    heights of instrument, station elevations, total distances, distance from the
    benchmark, error of misclosure, order of accuracy, and adjusted elevations.
    '''

    def __init__(self, data, level_type):
        '''
        Store the leveling dataframe and detected leveling type for computation.
        '''
        self.three_wire = '3-Wire'
        self.differential = 'Differential'
        self.HI_list = []
        self.elev_list = []

        # instead of relying on global notebook variables.
        self.data = data
        self.level_type = level_type

    def compute_elev(self):
        '''
        Compute height of instrument values and station elevations.

        For 3-wire leveling data, backsight and foresight readings are first
        averaged from the upper, middle, and lower wire readings. For both
        supported leveling types, the method computes the height of instrument
        and the resulting elevation for each row.
        '''
        self.bm_elev = float(input('Enter BM Elevation: '))
        self.current_elev = self.bm_elev

        if self.level_type == self.three_wire:
            for idx, row in self.data.iterrows():
                # average BS wire readings using built-in round()
                self.data.loc[idx, 'BS_Reading'] = round((
                    row['BS_upper'] + row['BS_middle'] + row['BS_lower']
                ) / 3, 3)

                # average FS wire readings using built-in round()
                self.data.loc[idx, 'FS_Readings'] = round((
                    row['FS_upper'] + row['FS_middle'] + row['FS_lower']
                ) / 3, 3)
        else:
            pass

        for idx, row in self.data.iterrows():
            # compute HI
            HI_value = self.current_elev + self.data.loc[idx, 'BS_Reading']
            self.HI_list.append(HI_value)

            # compute new elevation
            stat_elev = HI_value - self.data.loc[idx, 'FS_Readings']
            self.elev_list.append(stat_elev)

            self.current_elev = stat_elev

        self.data['Elevation'] = self.elev_list

        return self.data, self.HI_list, self.elev_list

    def misclosure(self):
        '''
        Compute loop distances, error of misclosure, and order of accuracy.

        For 3-wire data, sight distances are computed from the difference between
        upper and lower stadia readings. For differential data, existing distance
        columns are used. The method then computes total distance, misclosure,
        and accuracy classification using first, second, and third order limits.
        '''
        running_dist = 0
        if self.level_type == self.three_wire:
            for idx, row in self.data.iterrows():
                self.data.loc[idx, 'BS_Dist'] = (row['BS_upper'] - row['BS_lower']) * 100
                self.data.loc[idx, 'FS_Dist'] = (row['FS_upper'] - row['FS_lower']) * 100
                self.data.loc[idx, 'Total_Dist'] = self.data.loc[idx, 'BS_Dist'] + self.data.loc[idx, 'FS_Dist']

                running_dist += self.data.loc[idx, 'Total_Dist']
                self.data.loc[idx, 'Dist_from_BM'] = running_dist
        else:
            for idx, row in self.data.iterrows():
                self.data.loc[idx, 'Total_Dist'] = row['BS_Dist'] + row['FS_Dist']

                running_dist += self.data.loc[idx, 'Total_Dist']
                self.data.loc[idx, 'Dist_from_BM'] = running_dist

        tot_dist = self.data['Total_Dist'].sum()
        tot_bs = self.data['BS_Reading'].sum()
        tot_fs = self.data['FS_Readings'].sum()
        misclosure = round((tot_bs - tot_fs), 3)

        try:
            # convert meters to kilometers
            tot_dist_km = tot_dist / 1000

            allowable_first = 0.004 * (math.sqrt(tot_dist_km))
            allowable_second = 0.008 * (math.sqrt(tot_dist_km))
            allowable_third = 0.012 * (math.sqrt(tot_dist_km))

            abs_error = abs(misclosure)

            if abs_error <= allowable_first:
                order = 'First Order Accuracy'
            elif abs_error <= allowable_second:
                order = 'Second Order Accuracy'
            elif abs_error <= allowable_third:
                order = 'Third Order Accuracy'
            else:
                order = 'Rejected / Low Accuracy'

            return round(misclosure, 4), order

        except Exception as e:
            print(f'Error evaluating accuracy: {e}')
            return misclosure, 'Unknown'

    def adjusted_elev(self, misclosure):
        '''
        Apply proportional corrections to compute adjusted elevations.

        The correction is distributed by distance from the benchmark over the
        total loop distance. The adjusted elevations are saved in the Adj_Elev
        column of the dataframe.
        '''

        total_loop_dist = self.data['Total_Dist'].sum()

        for idx, row in self.data.iterrows():
            if total_loop_dist != 0:
                correction = (misclosure * -1) * (row['Dist_from_BM'] / total_loop_dist)
            else:
                correction = 0
            self.data.loc[idx, 'Adj_Elev'] = row['Elevation'] + correction

In [14]:
# Create FileHandler object to manage CSV input and detection
solver = LevelLoop(data, level_type)

# Read CSV file and get dataset + leveling type
data_2, HI_list, elev_list = solver.compute_elev()

data_2.head()

Enter BM Elevation:  100


,BS_point,BS_upper,BS_middle,BS_lower,FS_point,FS_upper,FS_middle,FS_lower,BS_Reading,FS_Readings,Elevation
0,BM1,0.89,0.70,0.51,TP 1,2.18,1.99,1.80,0.700,1.990,98.710
1,TP 1,1.66,1.53,1.40,TP 2,1.06,0.93,0.80,1.530,0.930,99.310
2,TP 2,1.65,1.57,1.50,TP 3,1.05,1.02,0.94,1.573,1.003,99.880
3,TP 3,1.74,1.66,1.58,TP 4,1.10,1.02,0.94,1.660,1.020,100.520
4,TP 4,1.91,1.82,1.73,TP 5,0.93,0.84,0.74,1.820,0.837,101.503


## Objectives 5-6: Report Export and Elevation Profile Graph


In [15]:
import matplotlib.pyplot as plt

class MakeReport:
    '''
    Export the final leveling computation report and elevation profile graph.

    The class receives the final dataframe, error of misclosure, and order of
    accuracy. It can save the report as either a text file or an HTML file, and
    it automatically creates an elevation profile graph using adjusted elevation
    values.
    '''

    def __init__(self, final_dataframe, error_val, acc_order):
        '''
        Store the final dataframe, misclosure value, and accuracy order.
        '''
        self.df = final_dataframe
        self.error = error_val
        self.order = acc_order

    def save_graph(self, base_name):
        '''
        Save an adjusted elevation profile graph as a PNG file.
        '''
        # Graph filename using the base name
        graph_filename = base_name + '_profile.png'

        try:
            plt.figure(figsize=(10, 5))
            x_vals = self.df['Dist_from_BM']
            y_vals = self.df['Adj_Elev']

            # Plot the main line
            plt.plot(x_vals, y_vals, marker='o', linestyle='-', color='darkred', label='Adjusted Elevation')

            # Area under the line
            plt.fill_between(x_vals, y_vals, color='red', alpha=0.3)

            # Combine the user's input name with "Elevation Profile"
            plt.title(f'{base_name} Elevation Profile')
            plt.xlabel('Distance from BM (m)')
            plt.ylabel('Elevation (m)')

            # Grid lines transparency
            plt.grid(True, alpha=0.5)
            plt.legend()

            plt.savefig(graph_filename, bbox_inches='tight')
            plt.close()

            print(f'Elevation Profile saved as {graph_filename}')

        except Exception as e:
            print('Oops, error saving graph image:', e)

    def save_to_txt(self):
        '''
        Ask for a project name and export the report as a text file."\
        '''
        raw_name = input('Type the base project name (e.g. Site_A): ')

        # In case they accidentally typed ".txt"
        base_name = raw_name.replace('.txt', '').replace('.html', '')

        # Create the new filename with _report attached
        filename = base_name + '_report.txt'

        try:
            text_file = open(filename, 'w')
            text_file.write("--- LEVELING COMPUTATION REPORT ---\n\n")
            text_file.write(self.df.to_string(index=False))
            text_file.write("\n\n--- MISCLOSURE AND ACCURACY SUMMARY ---\n")
            text_file.write(f"Error of Misclosure: {self.error}\n")
            text_file.write(f"Order of Accuracy: {self.order}\n")
            text_file.close()

            print('\nReport saved as', filename)

            # Saving Graph
            self.save_graph(base_name)

        except Exception as e:
            print('Oops, error saving text file:', e)

    def save_to_html(self):
        '''
        Ask for a project name and export the report as an HTML file.
        '''
        raw_name = input('Type the base project name (e.g. Site_A): ')

        base_name = raw_name.replace('.txt', '').replace('.html', '')

        # Create the new filename with _report attached
        filename = base_name + '_report.html'

        try:
            html_file = open(filename, 'w')
            html_file.write("<html>\n<head>\n<title>Leveling Report</title>\n")
            html_file.write("<style>table {border-collapse: collapse; width: 100%;} th, td {border: 1px solid black; padding: 8px; text-align: center;} body {font-family: Arial, sans-serif; margin: 40px;}</style>\n")
            html_file.write("</head>\n<body>\n")

            html_file.write("<h2>Leveling Computation Report</h2>\n")
            html_file.write(self.df.to_html(index=False, classes='table'))

            html_file.write("<h2>Misclosure and Accuracy Summary</h2>\n")
            html_file.write(f"<p><b>Error of Misclosure:</b> {self.error}</p>\n")
            html_file.write(f"<p><b>Order of Accuracy:</b> {self.order}</p>\n")

            html_file.write("</body>\n</html>\n")
            html_file.close()

            print('\nReport saved as', filename)

            # Saving Graph
            self.save_graph(base_name)

        except Exception as e:
            print('Oops, error saving html file:', e)

    def run_menu(self):
        '''
        Display the export menu and run the selected report export option.
        '''
        print("\n--- EXPORT MENU ---")
        print("Choose your report format (A profile graph will automatically be generated):")
        print("1 - Text file (.txt)")
        print("2 - HTML file (.html)")

        is_valid_choice = False
        while not is_valid_choice:
            user_choice = input("Enter 1 or 2: ")

            if user_choice == '1':
                self.save_to_txt()
                is_valid_choice = True
            elif user_choice == '2':
                self.save_to_html()
                is_valid_choice = True
            else:
                print("Invalid choice. Try again.")

In [16]:
# Calculate misclosure and accuracy
my_misclosure, my_accuracy = solver.misclosure()

# Calculate the adjusted elevations based on the misclosure
solver.adjusted_elev(my_misclosure)

# Create the report object and pass in the updated dataframe (data_2) and our calculated results
my_report = MakeReport(data_2, my_misclosure, my_accuracy)

# Run the export menu to let the user choose txt or html
my_report.run_menu()


--- EXPORT MENU ---
Choose your report format (A profile graph will automatically be generated):
1 - Text file (.txt)
2 - HTML file (.html)


Enter 1 or 2:  2
Type the base project name (e.g. Site_A):  Leveling-3wire



Report saved as Leveling-3wire_report.html
Elevation Profile saved as Leveling-3wire_profile.png
